In [8]:
import os
import time
import pydicom
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.model_selection import KFold
from sklearn.metrics import roc_auc_score

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import timm

# =============================================================================
# 1. CONFIGURATION & PATHS
# =============================================================================
COMPETITION_ROOT = Path("/kaggle/input/competitions/rsna-knee-abnormality-detection")
LLM_LABELS_CSV = Path("/kaggle/input/datasets/tnshpp/llm-targets/llm_labels_gemma-31b.csv")  # Update this path
OUTPUT_DIR = Path("./radimagenet_training")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

N_FOLDS = 4
SEED = 2026
EPOCHS = 15
BATCH_SIZE = 32
LR_HEAD = 1e-3
LR_BACKBONE = 1e-5
PATIENCE = 3  # Set desired patience threshold

TRAIN_BACKBONE = True  # Set to True to fine-tune the ResNet50 backbone
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

TARGETS = [
    "ACL", "MCL", "Medial Meniscus", "Lateral Meniscus", "Medial OA",
    "Lateral OA", "PF OA", "Effusion", "Synovitis", "Baker's",
    "Contusion", "Fracture"
]

# We extract 3-slice triplets (z-1, z, z+1) across 3 canonical planes
PLANES = ["Sagittal", "Coronal", "Axial"]



In [9]:


# =============================================================================
# 2. DATASET & PREPROCESSING
# =============================================================================
def load_and_preprocess_series(series_dir, crop_mm=140.0, out_size=256, n_slice=3):
    dcm_files = sorted(Path(series_dir).glob("*.dcm"))
    if not dcm_files: return None
    
    lo, hi = int(0.2 * (len(dcm_files) - 1)), int(0.8 * (len(dcm_files) - 1))
    idx = np.unique(np.linspace(lo, hi, n_slice).astype(int)) if hi > lo else np.array([len(dcm_files) // 2])
        
    while len(idx) < n_slice: idx = np.append(idx, idx[-1])
        
    planes_data = []
    px_spacing = 0.5
    for i in idx[:n_slice]:
        try:
            ds = pydicom.dcmread(str(dcm_files[i]), force=True)
            arr = ds.pixel_array.astype(np.float32)
            slope = float(getattr(ds, 'RescaleSlope', 1) or 1)
            intercept = float(getattr(ds, 'RescaleIntercept', 0) or 0)
            arr = arr * slope + intercept
            if getattr(ds, 'PixelSpacing', None) is not None:
                px_spacing = float(ds.PixelSpacing[0])
            planes_data.append(arr)
        except Exception: return None

    if not planes_data: return None

    vol = np.stack(planes_data)
    h, w = vol.shape[1:]
    want = int(round(crop_mm / px_spacing))
    if 16 < want < min(h, w):
        cy, cx = h // 2, w // 2
        half = want // 2
        vol = vol[:, max(0, cy - half):cy + half, max(0, cx - half):cx + half]

    lo_v, hi_v = np.percentile(vol, [1, 99])
    vol = np.clip((vol - lo_v) / max(hi_v - lo_v, 1e-6), 0, 1)

    t = torch.from_numpy(np.ascontiguousarray(vol)).unsqueeze(0)
    t = F.interpolate(t, size=(out_size, out_size), mode='bilinear', align_corners=False)
    return (t.squeeze(0) * 255.0).round().clamp(0, 255).to(torch.uint8)

class RadImageNetDataset(Dataset):
    def __init__(self, df, series_df, data_dir):
        self.df = df.reset_index(drop=True)
        self.series_df = series_df
        self.data_dir = Path(data_dir)

    def __len__(self): return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        study_uid = str(row["StudyInstanceUID"])
        study_series = self.series_df[self.series_df["StudyInstanceUID"] == study_uid]
        
        img_tensor = torch.zeros((len(PLANES), 3, 256, 256), dtype=torch.uint8)
        mask = torch.zeros(len(PLANES), dtype=torch.float32)
        
        used_series = set()
        for p_idx, plane in enumerate(PLANES):
            cand = study_series[(study_series["Anatomical_Plane"] == plane) & 
                                (~study_series["SeriesInstanceUID"].isin(used_series))]
            if not cand.empty:
                chosen = cand.sort_values("n_slices", ascending=False).iloc[0] if "n_slices" in cand.columns else cand.iloc[0]
                s_uid = str(chosen["SeriesInstanceUID"])
                used_series.add(s_uid)
                
                s_path = self.data_dir / study_uid / s_uid
                img = load_and_preprocess_series(s_path, crop_mm=140.0, out_size=256)
                if img is not None:
                    img_tensor[p_idx] = img
                    mask[p_idx] = 1.0

        labels = torch.tensor(row[TARGETS].values.astype(np.float32))
        return img_tensor, mask, labels

# =============================================================================
# 3. RADIMAGENET ARCHITECTURE
# =============================================================================
class RadHeadModule(nn.Module):
    def __init__(self, in_dim=2048, hidden_dim=512, num_classes=12):
        super().__init__()
        self.proj = nn.Sequential(nn.Linear(in_dim, hidden_dim), nn.ReLU(), nn.Dropout(0.2))
        self.classifier = nn.Linear(hidden_dim, num_classes)

    def forward(self, x): return self.classifier(self.proj(x))

class Stage3RadModel(nn.Module):
    def __init__(self, train_backbone=TRAIN_BACKBONE):
        super().__init__()
        self.backbone = timm.create_model('resnet50', pretrained=True, num_classes=0, global_pool='avg')
        for p in self.backbone.parameters(): p.requires_grad = train_backbone
        self.head = RadHeadModule(in_dim=self.backbone.num_features, hidden_dim=512, num_classes=len(TARGETS))
        self.register_buffer('mean', torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1))
        self.register_buffer('std', torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1))

    def forward(self, imgs, mask):
        B, P = imgs.shape[:2]
        x = imgs.reshape(B * P, *imgs.shape[2:]).float().div_(255.0)
        x = (x - self.mean) / self.std
        
        features = self.backbone(x) if any(p.requires_grad for p in self.backbone.parameters()) else self.backbone(x).detach()
        features = features.reshape(B, P, -1)
        
        mask_expanded = mask.unsqueeze(-1)
        pooled_features = (features * mask_expanded).sum(dim=1) / mask_expanded.sum(dim=1).clamp(min=1.0)
        return self.head(pooled_features)

# =============================================================================
# 4. TRAINING & EVALUATION
# =============================================================================
def train_epoch(model, loader, optimizer, criterion, scaler):
    model.train()
    total_loss = 0.0
    for imgs, mask, labels in loader:
        imgs, mask, labels = imgs.to(DEVICE), mask.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        with torch.amp.autocast('cuda'):
            loss = criterion(model(imgs, mask), labels)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        total_loss += loss.item() * len(labels)
    return total_loss / len(loader.dataset)

@torch.no_grad()
def validate_epoch(model, loader, criterion):
    model.eval()
    total_loss = 0.0
    all_preds, all_targets = [], []
    for imgs, mask, labels in loader:
        imgs, mask, labels = imgs.to(DEVICE), mask.to(DEVICE), labels.to(DEVICE)
        with torch.amp.autocast('cuda'):
            logits = model(imgs, mask)
            loss = criterion(logits, labels)
            probs = torch.sigmoid(logits)
        total_loss += loss.item() * len(labels)
        all_preds.append(probs.cpu().numpy())
        all_targets.append(labels.cpu().numpy())
        
    y_pred, y_true = np.vstack(all_preds), np.vstack(all_targets)
    y_true_binary = (y_true >= 0.5).astype(int)
    aucs = [roc_auc_score(y_true_binary[:, i], y_pred[:, i]) for i in range(len(TARGETS)) if len(np.unique(y_true_binary[:, i])) > 1]
    return total_loss / len(loader.dataset), np.mean(aucs) if aucs else 0.5, y_pred

# =============================================================================
# 5. PIPELINE EXECUTION
# =============================================================================


def run_pipeline():
    print(f"Initializing RadImageNet Pipeline on {DEVICE}")

    # 1. Load Golden Benchmark
    golden_df = (
        pd.read_csv(COMPETITION_ROOT / "train.csv")
        .dropna(subset=TARGETS)
        .reset_index(drop=True)
    )
    golden_uids = set(golden_df["StudyInstanceUID"].astype(str))

    # 2. Load LLM Data and Exclude Golden Benchmark
    llm_df = pd.read_csv(LLM_LABELS_CSV)
    for tar in TARGETS:
        llm_df[tar] = llm_df[f"{tar}_y"]

    llm_df = (
        llm_df[["StudyInstanceUID"] + TARGETS]
        .dropna(subset=TARGETS)
        .reset_index(drop=True)
    )

    # Strictly remove Golden Benchmark studies from training set to avoid data leakage
    llm_df = llm_df[
        ~llm_df["StudyInstanceUID"].astype(str).isin(golden_uids)
    ].reset_index(drop=True)

    series_df = pd.read_csv(COMPETITION_ROOT / "train_series.csv")
    series_df["StudyInstanceUID"] = series_df["StudyInstanceUID"].astype(str)
    series_df["SeriesInstanceUID"] = series_df["SeriesInstanceUID"].astype(str)

    golden_ds = RadImageNetDataset(
        golden_df, series_df, COMPETITION_ROOT / "train_series"
    )
    golden_loader = DataLoader(
        golden_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=4
    )

    oof_predictions = np.zeros((len(llm_df), len(TARGETS)), dtype=np.float32)
    kf = KFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
    golden_preds_folds = []

    for fold, (train_idx, val_idx) in enumerate(kf.split(llm_df)):
        print(
            f"\n==================== FOLD {fold + 1}/{N_FOLDS} ===================="
        )
        train_loader = DataLoader(
            RadImageNetDataset(
                llm_df.iloc[train_idx],
                series_df,
                COMPETITION_ROOT / "train_series",
            ),
            batch_size=BATCH_SIZE,
            shuffle=True,
            num_workers=4,
        )
        val_loader = DataLoader(
            RadImageNetDataset(
                llm_df.iloc[val_idx],
                series_df,
                COMPETITION_ROOT / "train_series",
            ),
            batch_size=BATCH_SIZE,
            shuffle=False,
            num_workers=4,
        )

        model = Stage3RadModel(train_backbone=TRAIN_BACKBONE).to(DEVICE)
        criterion = nn.BCEWithLogitsLoss()

        param_groups = [{"params": model.head.parameters(), "lr": LR_HEAD}]
        if TRAIN_BACKBONE:
            param_groups.append({
                "params": [
                    p
                    for n, p in model.backbone.named_parameters()
                    if p.requires_grad
                ],
                "lr": LR_BACKBONE,
            })

        optimizer = torch.optim.AdamW(param_groups, weight_decay=0.02)
        scaler = torch.amp.GradScaler("cuda")
        best_auc, best_preds = 0.0, None
        patience_counter = 0

        for epoch in range(1, EPOCHS + 1):
            t0 = time.time()
            train_loss = train_epoch(
                model, train_loader, optimizer, criterion, scaler
            )
            val_loss, val_auc, val_preds = validate_epoch(
                model, val_loader, criterion
            )
            print(
                f"Ep {epoch:02d}/{EPOCHS:02d} | Train Loss: {train_loss:.4f} |"
                f" Val AUC: {val_auc:.4f} | Time: {time.time() - t0:.1f}s"
            )

            if val_auc > best_auc:
                best_auc = val_auc
                best_preds = val_preds
                patience_counter = 0
                torch.save(
                    model.state_dict(),
                    OUTPUT_DIR / f"rad_heads_fold{fold}.pt",
                )
            else:
                patience_counter += 1
                if patience_counter >= PATIENCE:
                    print(
                        f"Early stopping triggered at epoch {epoch} (No"
                        f" improvement for {PATIENCE} epochs)."
                    )
                    break

        oof_predictions[val_idx] = best_preds

        # Evaluate Golden Benchmark for this fold
        model.load_state_dict(
            torch.load(OUTPUT_DIR / f"rad_heads_fold{fold}.pt")
        )
        _, _, g_pred = validate_epoch(model, golden_loader, criterion)
        golden_preds_folds.append(g_pred)

    # Save OOF for Stage 3 Calibrator
    np.savez_compressed(OUTPUT_DIR / "rad_e10_raw.npz", values=oof_predictions)

    # Golden Benchmark Report
    print("\n" + "=" * 70)
    print(" RADIOLOGIST GOLDEN BENCHMARK PERFORMANCE REPORT (58 STUDIES) ")
    print("=" * 70)
    golden_ensemble_preds = np.mean(golden_preds_folds, axis=0)
    golden_targets = golden_df[TARGETS].values.astype(np.float32)
    y_true_binary = (golden_targets >= 0.5).astype(int)

    metrics = []
    for i, target in enumerate(TARGETS):
        auc = (
            roc_auc_score(y_true_binary[:, i], golden_ensemble_preds[:, i])
            if len(np.unique(y_true_binary[:, i])) > 1
            else 0.5
        )
        metrics.append({"Target": target, "ROC-AUC": auc})

    report_df = pd.DataFrame(metrics)
    print(report_df.to_string(index=False))
    print("-" * 70)
    print(
        " GOLDEN BENCHMARK MACRO ROC-AUC :"
        f" {report_df['ROC-AUC'].mean():.4f}"
    )
    report_df.to_csv(OUTPUT_DIR / "golden_benchmark_metrics.csv", index=False)

In [10]:
run_pipeline()

Initializing RadImageNet Pipeline on cuda

==================== FOLD 1/5 ====================
Ep 01/10 | Train Loss: 0.5847 | Val AUC: 0.6094 | Time: 140.6s
Ep 02/10 | Train Loss: 0.5704 | Val AUC: 0.6464 | Time: 141.4s
Ep 03/10 | Train Loss: 0.5621 | Val AUC: 0.6636 | Time: 140.9s
Ep 04/10 | Train Loss: 0.5551 | Val AUC: 0.6738 | Time: 141.5s
Ep 05/10 | Train Loss: 0.5497 | Val AUC: 0.6775 | Time: 140.7s
Ep 06/10 | Train Loss: 0.5418 | Val AUC: 0.6811 | Time: 142.2s
Ep 07/10 | Train Loss: 0.5371 | Val AUC: 0.6843 | Time: 142.0s
Ep 08/10 | Train Loss: 0.5305 | Val AUC: 0.6829 | Time: 142.1s


KeyboardInterrupt: 